# Cell 1: Import Modul dan Konfigurasi
Sel pertama digunakan untuk memuat semua library yang diperlukan dan mendefinisikan konfigurasi awal menggunakan @dataclass. Pendekatan ini memusatkan seluruh parameter statis agar mudah disesuaikan tanpa perlu mencari variabel di dalam logika utama.

In [10]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

@dataclass
class Config:
    train1_path: str = 'train1.csv'
    train2_path: str = 'train2.csv'
    test_path: str = 'test.csv'
    sub_path: str = 'sample_submission.csv'
    original_data_path: str = 'credit_risk_dataset.csv'
    n_splits: int = 5
    random_state: int = 42
    verbose_eval: int = 100
    n_trials: int = 40
    gpu_ram_fraction: float = 0.5

# Cell 2: Modul Praproses Data
Kelas DataPreprocessor diterapkan menggunakan dekorator @staticmethod karena fungsi-fungsi di dalamnya bersifat utilitas murni dan tidak perlu menyimpan state internal. Logika berfokus pada ekstraksi fitur waktu (tahun, bulan, hari) dari kolom date serta mengubah tipe data kategorikal pada kolom family menjadi representasi numerik menggunakan LabelEncoder.

In [11]:
class DataPreprocessor:
    @staticmethod
    def load_and_preprocess(config: Config):
        train1 = pd.read_csv(config.train1_path).drop(columns=['id', 'sample_id'], errors='ignore')
        train2 = pd.read_csv(config.train2_path).drop(columns=['id', 'sample_id'], errors='ignore')
        test = pd.read_csv(config.test_path)
        sub = pd.read_csv(config.sub_path)
        orig = pd.read_csv(config.original_data_path)

        train = pd.concat([train1, train2, orig], ignore_index=True).drop_duplicates().reset_index(drop=True)
        
        train['is_test'] = 0
        test['is_test'] = 1
        test['loan_status'] = np.nan
        
        all_data = pd.concat([train, test], ignore_index=True)
        
        all_data['person_age'] = all_data['person_age'].clip(upper=85)
        all_data['person_emp_length'] = all_data['person_emp_length'].clip(upper=40)
        all_data['person_emp_length'] = all_data['person_emp_length'].fillna(all_data['person_emp_length'].median())
        
        all_data['loan_int_rate'] = all_data.groupby('loan_grade')['loan_int_rate'].transform(lambda x: x.fillna(x.median()))
        all_data['loan_int_rate'] = all_data['loan_int_rate'].fillna(all_data['loan_int_rate'].median())

        all_data['person_income_log'] = np.log1p(all_data['person_income'])
        all_data['loan_amnt_log'] = np.log1p(all_data['loan_amnt'])

        all_data['emp_to_age_ratio'] = all_data['person_emp_length'] / all_data['person_age']
        all_data['loan_to_income_ratio'] = all_data['loan_amnt'] / (all_data['person_income'] + 1)
        
        monthly_rate = all_data['loan_int_rate'] / 1200
        all_data['monthly_payment'] = all_data['loan_amnt'] * monthly_rate / (1 - (1 + monthly_rate)**(-36))
        all_data['payment_to_income_ratio'] = all_data['monthly_payment'] / ((all_data['person_income'] / 12) + 1)
        
        all_data['home_and_intent'] = all_data['person_home_ownership'].astype(str) + "_" + all_data['loan_intent'].astype(str)
        all_data['grade_and_default'] = all_data['loan_grade'].astype(str) + "_" + all_data['cb_person_default_on_file'].astype(str)

        categorical_cols = [
            'person_home_ownership', 'loan_intent', 'loan_grade', 
            'cb_person_default_on_file', 'home_and_intent', 'grade_and_default'
        ]
        
        for col in categorical_cols:
            all_data[col] = all_data[col].fillna('Missing').astype(str)

        final_train = all_data[all_data['is_test'] == 0].drop(columns=['is_test'])
        final_test = all_data[all_data['is_test'] == 1].drop(columns=['is_test', 'loan_status'])
        
        return final_train, final_test, sub, categorical_cols

# Cell 3: Modul Fine-Tuning (Optuna)
Kelas ini bertanggung jawab untuk mengeksekusi pencarian ruang parameter. State internal tidak disimpan dalam jangka panjang, fungsi hanya mengembalikan dictionary berisi parameter terbaik. Optuna diinstruksikan untuk meminimalkan nilai fungsi objektif (RMSE).

In [12]:
class HyperparameterTuner:
    @staticmethod
    def objective_lgb(trial, X: pd.DataFrame, y: pd.Series, config: Config):
        params = {
            'objective': 'binary',
            'metric': 'auc',
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 5, 12),
            'num_leaves': trial.suggest_int('num_leaves', 31, 150),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 5),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
            'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 3.5),
            'seed': config.random_state,
            'verbose': -1
        }

        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=config.random_state)
        auc_scores = []

        for train_idx, val_idx in skf.split(X, y):
            X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
            X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
            
            train_data = lgb.Dataset(X_train, label=y_train)
            val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
            
            model = lgb.train(
                params,
                train_data,
                num_boost_round=500,
                valid_sets=[val_data],
                callbacks=[lgb.early_stopping(stopping_rounds=40, verbose=False)]
            )
            
            preds = model.predict(X_val, num_iteration=model.best_iteration)
            auc_scores.append(roc_auc_score(y_val, preds))

        return np.mean(auc_scores)

    @staticmethod
    def objective_cat(trial, X: pd.DataFrame, y: pd.Series, cat_features: list, config: Config):
        params = {
            'task_type': 'GPU',
            'gpu_ram_part': config.gpu_ram_fraction,
            'iterations': 500,
            'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
            'depth': trial.suggest_int('depth', 4, 10),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
            'random_strength': trial.suggest_float('random_strength', 0.1, 10.0),
            'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
            'eval_metric': 'AUC',
            'random_seed': config.random_state,
            'verbose': False
        }

        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=config.random_state)
        auc_scores = []

        for train_idx, val_idx in skf.split(X, y):
            X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
            X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
            
            train_pool = Pool(X_train, y_train, cat_features=cat_features)
            val_pool = Pool(X_val, y_val, cat_features=cat_features)
            
            model = CatBoostClassifier(**params)
            model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=40, use_best_model=True)
            
            preds = model.predict_proba(X_val)[:, 1]
            auc_scores.append(roc_auc_score(y_val, preds))

        return np.mean(auc_scores)

    @staticmethod
    def run_tuning(X: pd.DataFrame, y: pd.Series, cat_features: list, config: Config) -> tuple:
        optuna.logging.set_verbosity(optuna.logging.INFO)
        
        print("--- Memulai Tuning LightGBM ---")
        X_lgb = X.copy()
        for col in cat_features:
            X_lgb[col] = X_lgb[col].astype('category')
            
        study_lgb = optuna.create_study(direction='maximize')
        study_lgb.optimize(lambda trial: HyperparameterTuner.objective_lgb(trial, X_lgb, y, config), n_trials=config.n_trials)
        print(f"[LightGBM] ROC AUC Terbaik: {study_lgb.best_value:.4f}")
        
        print("\n--- Memulai Tuning CatBoost ---")
        study_cat = optuna.create_study(direction='maximize')
        study_cat.optimize(lambda trial: HyperparameterTuner.objective_cat(trial, X, y, cat_features, config), n_trials=config.n_trials)
        print(f"[CatBoost] ROC AUC Terbaik: {study_cat.best_value:.4f}")
        
        return study_lgb.best_params, study_cat.best_params

# Cell 4: Modul Pelatihan LightGBM
Kelas LightGBMModel melakukan instansiasi menggunakan __init__ untuk menyimpan daftar model dari setiap lipatan (fold). Proses training menerapkan K-Fold Cross Validation, di mana data dipecah menjadi beberapa bagian (misalnya 5). Model dilatih pada 4 bagian dan divalidasi pada 1 bagian yang tersisa secara bergantian. Evaluasi diukur menggunakan metrik RMSE (Root Mean Squared Error).

In [13]:
class EnsembleModels:
    def __init__(self, config: Config, best_lgb_params: dict, best_cat_params: dict, cat_features: list):
        self.config = config
        self.cat_features = cat_features
        
        self.lgb_params = {
            'objective': 'binary',
            'metric': 'auc',
            'seed': self.config.random_state,
            'verbose': -1
        }
        self.lgb_params.update(best_lgb_params)
        
        self.cat_params = {
            'task_type': 'GPU',
            'gpu_ram_part': self.config.gpu_ram_fraction,
            'iterations': 1500,
            'eval_metric': 'AUC',
            'random_seed': self.config.random_state,
            'verbose': False
        }
        self.cat_params.update(best_cat_params)

    def train_and_predict(self, X: pd.DataFrame, y: pd.Series, X_test: pd.DataFrame):
        skf = StratifiedKFold(n_splits=self.config.n_splits, shuffle=True, random_state=self.config.random_state)
        
        lgb_oof = np.zeros(len(X))
        cat_oof = np.zeros(len(X))
        lgb_preds = np.zeros(len(X_test))
        cat_preds = np.zeros(len(X_test))
        
        X_lgb = X.copy()
        X_test_lgb = X_test.copy()
        for col in self.cat_features:
            X_lgb[col] = X_lgb[col].astype('category')
            X_test_lgb[col] = X_test_lgb[col].astype('category')
            
        print("Mengeksekusi Pelatihan K-Fold Cross Validation...")
        for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
            print(f"\n--- Melatih Fold {fold+1}/{self.config.n_splits} ---")
            
            X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
            X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
            
            X_tr_lgb, X_va_lgb = X_lgb.iloc[train_idx], X_lgb.iloc[val_idx]
            
            # --- Pelatihan LightGBM ---
            lgb_train = lgb.Dataset(X_tr_lgb, label=y_tr)
            lgb_val = lgb.Dataset(X_va_lgb, label=y_va, reference=lgb_train)
            
            model_lgb = lgb.train(
                self.lgb_params,
                lgb_train,
                num_boost_round=1500,
                valid_sets=[lgb_val],
                callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
            )
            lgb_oof[val_idx] = model_lgb.predict(X_va_lgb, num_iteration=model_lgb.best_iteration)
            lgb_preds += model_lgb.predict(X_test_lgb, num_iteration=model_lgb.best_iteration) / self.config.n_splits
            
            # --- Pelatihan CatBoost ---
            cat_train = Pool(X_tr, y_tr, cat_features=self.cat_features)
            cat_val = Pool(X_va, y_va, cat_features=self.cat_features)
            
            model_cat = CatBoostClassifier(**self.cat_params)
            model_cat.fit(cat_train, eval_set=cat_val, early_stopping_rounds=50, use_best_model=True)
            
            cat_oof[val_idx] = model_cat.predict_proba(X_va)[:, 1]
            cat_preds += model_cat.predict_proba(Pool(X_test, cat_features=self.cat_features))[:, 1] / self.config.n_splits
            
            print(f"LGBM AUC: {roc_auc_score(y_va, lgb_oof[val_idx]):.4f} | CatBoost AUC: {roc_auc_score(y_va, cat_oof[val_idx]):.4f}")

        print("\n=== RINGKASAN EVALUASI OOF (Out-Of-Fold) ===")
        print(f"Skor LightGBM Murni : {roc_auc_score(y, lgb_oof):.5f}")
        print(f"Skor CatBoost Murni : {roc_auc_score(y, cat_oof):.5f}")
        
        ensemble_oof = (lgb_oof * 0.5) + (cat_oof * 0.5)
        print(f"Skor ENSEMBLE (50:50): {roc_auc_score(y, ensemble_oof):.5f}")
        
        ensemble_preds = (lgb_preds * 0.5) + (cat_preds * 0.5)
        return ensemble_preds

# Cell 5: Eksekusi Utama dan Pembuatan Submission
Sel terakhir menggabungkan seluruh kelas modular yang telah didefinisikan. Data diproses, fitur dipisahkan dari target, lalu diteruskan ke dalam kelas pemodelan. Prediksi akhir dihitung melalui rata-rata (ensemble) dari seluruh model yang terbentuk di tahapan validasi K-Fold. Fungsi np.clip digunakan untuk mencegah nilai prediksi akhir berada di bawah nol, karena angka penjualan (sales) tidak mungkin bernilai negatif.

In [14]:
config = Config()

print("1. Memuat seluruh dataset dan memproses rekayasa fitur...")
train_df, test_df, sample_sub_df, cat_features = DataPreprocessor.load_and_preprocess(config)

features = [col for col in train_df.columns if col not in ['id', 'sample_id', 'loan_status']]
target = 'loan_status'

X = train_df[features]
y = train_df[target]
X_test = test_df[features]

print(f"Total Baris Pelatihan: {len(X)}")
print(f"Total Fitur: {len(features)}")

print("\n2. Memulai proses Optimasi Parameter (Optuna)...")
best_lgb_params, best_cat_params = HyperparameterTuner.run_tuning(X, y, cat_features, config)

print("\n3. Memulai Pelatihan Model Ensemble dengan Parameter Optimal...")
ensemble_trainer = EnsembleModels(config, best_lgb_params, best_cat_params, cat_features)
final_predictions = ensemble_trainer.train_and_predict(X, y, X_test)

print("\n4. Menyusun fail submission.csv dengan format probabilitas...")
id_col = sample_sub_df.columns[0]
target_col = sample_sub_df.columns[1]

test_id_col = 'sample_id' if 'sample_id' in test_df.columns else ('id' if 'id' in test_df.columns else test_df.columns[0])

final_sub = pd.DataFrame({
    id_col: test_df[test_id_col].values,
    target_col: final_predictions
})
final_sub.to_csv('submission_ensemble_optuna.csv', index=False)

print("\n[SELESAI] Fail 'submission_ensemble_optuna.csv' berhasil dibuat.")

1. Memuat seluruh dataset dan memproses rekayasa fitur...


[I 2026-06-05 12:51:13,422] A new study created in memory with name: no-name-dd31761d-d0e3-49a3-819a-f9038a1299e2


Total Baris Pelatihan: 135044
Total Fitur: 19

2. Memulai proses Optimasi Parameter (Optuna)...
--- Memulai Tuning LightGBM ---


[I 2026-06-05 12:51:20,200] Trial 0 finished with value: 0.9477365500226854 and parameters: {'learning_rate': 0.013302371104123132, 'max_depth': 11, 'num_leaves': 67, 'feature_fraction': 0.6244892721302685, 'bagging_fraction': 0.8036237703485617, 'bagging_freq': 4, 'min_child_samples': 21, 'scale_pos_weight': 2.1841768108233897}. Best is trial 0 with value: 0.9477365500226854.
[I 2026-06-05 12:51:26,348] Trial 1 finished with value: 0.9477586397615886 and parameters: {'learning_rate': 0.018990148191209474, 'max_depth': 8, 'num_leaves': 50, 'feature_fraction': 0.744988275922839, 'bagging_fraction': 0.9227172924049604, 'bagging_freq': 4, 'min_child_samples': 80, 'scale_pos_weight': 2.9218501735372957}. Best is trial 1 with value: 0.9477586397615886.
[I 2026-06-05 12:51:32,401] Trial 2 finished with value: 0.9469962609009294 and parameters: {'learning_rate': 0.01837914888478044, 'max_depth': 7, 'num_leaves': 147, 'feature_fraction': 0.6196365062174134, 'bagging_fraction': 0.86474742366036

[LightGBM] ROC AUC Terbaik: 0.9542

--- Memulai Tuning CatBoost ---


...

LGBM AUC: 0.9585 | CatBoost AUC: 0.9535

--- Melatih Fold 4/5 ---


Default metric period is 5 because AUC is/are not implemented for GPU


LGBM AUC: 0.9544 | CatBoost AUC: 0.9493

--- Melatih Fold 5/5 ---


Default metric period is 5 because AUC is/are not implemented for GPU


LGBM AUC: 0.9590 | CatBoost AUC: 0.9536

=== RINGKASAN EVALUASI OOF (Out-Of-Fold) ===
Skor LightGBM Murni : 0.95699
Skor CatBoost Murni : 0.95241
Skor ENSEMBLE (50:50): 0.95688

4. Menyusun fail submission.csv dengan format probabilitas...

[SELESAI] Fail 'submission_ensemble_optuna.csv' berhasil dibuat.
